# Golden Customer Record using Medallion Architecture

This notebook creates a Golden Customer Record from two source systems:

- CRM customers
- Transaction customers

## Architecture

### Bronze
Raw ingestion of source files without business transformations.

### Silver
Data standardization, quality handling, deduplication, and customer reconciliation.

### Gold
Final curated Golden Customer Record for downstream analytics and business use.

In [ ]:
!pip -q install pyspark==3.5.1

In [15]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("golden_customer_record")
    .master("local[*]")
    .getOrCreate()
)

crm_path = "/content/crm_customers.csv"
txn_path = "/content/transaction_customers.csv"

print("Spark started")

Spark started


## Bronze Layer — Raw ingestion

In this layer we read the source CSV files exactly as received.

Purpose:
- preserve source structure
- inspect raw row counts
- establish the input data boundary

In [16]:
crm_bronze = spark.read.option("header", True).csv(crm_path)
txn_bronze = spark.read.option("header", True).csv(txn_path)

print("CRM bronze rows:", crm_bronze.count())
print("TXN bronze rows:", txn_bronze.count())

print("CRM columns:", crm_bronze.columns)
print("TXN columns:", txn_bronze.columns)

crm_bronze.show(5, truncate=False)
txn_bronze.show(5, truncate=False)

CRM bronze rows: 108
TXN bronze rows: 80
CRM columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'address', 'city', 'country', 'registration_date', 'last_updated']
TXN columns: ['transaction_id', 'customer_email', 'first_name', 'last_name', 'phone', 'shipping_address', 'city', 'country', 'purchase_date']
+-----------+----------+---------+--------------------------+-------------+---------------+----------+-------+-----------------+------------+
|customer_id|first_name|last_name|email                     |phone        |address        |city      |country|registration_date|last_updated|
+-----------+----------+---------+--------------------------+-------------+---------------+----------+-------+-----------------+------------+
|CRM00001   |Matthew   |Brown    |matthew.brown@ymail.com   |NULL         |410 Oak Ave    |Manchester|UK     |2018-12-23       |2024-08-27  |
|CRM00002   |Charles   |Lee      |charles.lee@ymail.com     |+336598035557|435 Second Ave |Paris     |France

## Silver Layer — Standardization and cleansing

In this layer we:
- normalize email, phone, text, and dates
- remove exact duplicates
- create fallback business keys
- prepare the data for matching across systems

In [17]:
def parse_mixed_date(col_name):
    return F.coalesce(
        F.to_date(F.col(col_name), "yyyy-MM-dd"),
        F.to_date(F.col(col_name), "yyyyMMdd")
    )

def norm_email(col_name):
    return F.when(F.trim(F.col(col_name)) == "", F.lit(None)) \
            .otherwise(F.lower(F.trim(F.col(col_name))))

def norm_phone(col_name):
    return F.when(F.trim(F.col(col_name)) == "", F.lit(None)) \
            .otherwise(F.regexp_replace(F.col(col_name), r"[^0-9]", ""))

def norm_text(col_name):
    return F.when(F.trim(F.col(col_name)) == "", F.lit(None)) \
            .otherwise(F.initcap(F.trim(F.col(col_name))))

def norm_country(col_name):
    return F.when(F.trim(F.col(col_name)) == "", F.lit(None)) \
            .otherwise(F.upper(F.trim(F.col(col_name))))

crm_silver_raw = (
    crm_bronze
    .dropDuplicates()
    .withColumn("crm_customer_id", F.col("customer_id"))
    .withColumn("email_normalized", norm_email("email"))
    .withColumn("phone_normalized", norm_phone("phone"))
    .withColumn("first_name_clean", norm_text("first_name"))
    .withColumn("last_name_clean", norm_text("last_name"))
    .withColumn("address_clean", norm_text("address"))
    .withColumn("city_clean", norm_text("city"))
    .withColumn("country_clean", norm_country("country"))
    .withColumn("registration_date_clean", parse_mixed_date("registration_date"))
    .withColumn("last_updated_clean", parse_mixed_date("last_updated"))
    .withColumn("source_recency", F.greatest("registration_date_clean", "last_updated_clean"))
    .withColumn(
        "fallback_key",
        F.concat_ws(
            "|",
            F.lower(F.coalesce(F.col("first_name_clean"), F.lit(""))),
            F.lower(F.coalesce(F.col("last_name_clean"), F.lit(""))),
            F.lower(F.coalesce(F.col("city_clean"), F.lit(""))),
            F.lower(F.coalesce(F.col("country_clean"), F.lit("")))
        )
    )
)

txn_silver_raw = (
    txn_bronze
    .dropDuplicates()
    .withColumn("transaction_email", F.col("customer_email"))
    .withColumn("email_normalized", norm_email("customer_email"))
    .withColumn("phone_normalized", norm_phone("phone"))
    .withColumn("first_name_clean", norm_text("first_name"))
    .withColumn("last_name_clean", norm_text("last_name"))
    .withColumn("address_clean", norm_text("shipping_address"))
    .withColumn("city_clean", norm_text("city"))
    .withColumn("country_clean", norm_country("country"))
    .withColumn("purchase_date_clean", parse_mixed_date("purchase_date"))
    .withColumn("source_recency", F.col("purchase_date_clean"))
    .withColumn(
        "fallback_key",
        F.concat_ws(
            "|",
            F.lower(F.coalesce(F.col("first_name_clean"), F.lit(""))),
            F.lower(F.coalesce(F.col("last_name_clean"), F.lit(""))),
            F.lower(F.coalesce(F.col("city_clean"), F.lit(""))),
            F.lower(F.coalesce(F.col("country_clean"), F.lit("")))
        )
    )
)

print("CRM silver raw:", crm_silver_raw.count())
print("TXN silver raw:", txn_silver_raw.count())

crm_silver_raw.select(
    "crm_customer_id", "email_normalized", "phone_normalized",
    "first_name_clean", "city_clean", "country_clean"
).show(5, truncate=False)

txn_silver_raw.select(
    "transaction_id", "email_normalized", "phone_normalized",
    "first_name_clean", "city_clean", "country_clean"
).show(5, truncate=False)

CRM silver raw: 100
TXN silver raw: 80
+---------------+--------------------------+----------------+----------------+----------+-------------+
|crm_customer_id|email_normalized          |phone_normalized|first_name_clean|city_clean|country_clean|
+---------------+--------------------------+----------------+----------------+----------+-------------+
|CRM00087       |william.white@lmail.com   |332932872527    |William         |Manchester|FRANCE       |
|CRM00034       |matthew.martin@ymail.com  |333518711090    |Matthew         |Glasgow   |FRANCE       |
|CRM00036       |charles.anderson@omail.com|334659695050    |Charles         |Glasgow   |FRANCE       |
|CRM00069       |michael.anderson@email.com|445062291753    |Michael         |London    |UK           |
|CRM00044       |james.williams@email.com  |338143231672    |James           |Paris     |FRANCE       |
+---------------+--------------------------+----------------+----------------+----------+-------------+
only showing top 5 rows



## Silver Layer — Deduplication and reconciliation

In this layer we:
- keep the most recent source record per customer candidate
- match customers across systems using:
  1. exact email match
  2. fallback match on name + city + country only when email is missing and the fallback key is unique
- build transaction-level customer metrics

In [21]:
crm_window = Window.partitionBy(
    F.when(F.col("email_normalized").isNotNull(), F.col("email_normalized"))
     .otherwise(F.col("fallback_key"))
).orderBy(F.col("source_recency").desc_nulls_last(), F.col("crm_customer_id").asc())

txn_window = Window.partitionBy(
    F.when(F.col("email_normalized").isNotNull(), F.col("email_normalized"))
     .otherwise(F.col("fallback_key"))
).orderBy(F.col("source_recency").desc_nulls_last(), F.col("transaction_id").asc())

crm_silver = (
    crm_silver_raw
    .withColumn("rn", F.row_number().over(crm_window))
    .filter("rn = 1")
    .drop("rn")
)

txn_silver = (
    txn_silver_raw
    .withColumn("rn", F.row_number().over(txn_window))
    .filter("rn = 1")
    .drop("rn")
)

txn_metrics = (
    txn_silver_raw
    .groupBy("email_normalized", "fallback_key")
    .agg(
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.min("purchase_date_clean").alias("first_purchase_date"),
        F.max("purchase_date_clean").alias("last_purchase_date")
    )
)

email_matches = (
    crm_silver.alias("crm")
    .join(
        txn_silver.alias("txn"),
        F.col("crm.email_normalized") == F.col("txn.email_normalized"),
        "inner"
    )
    .filter(F.col("crm.email_normalized").isNotNull())
)

crm_email_matched = email_matches.select(F.col("crm.crm_customer_id").alias("crm_customer_id")).distinct()
txn_email_matched = email_matches.select(F.col("txn.transaction_id").alias("transaction_id")).distinct()

crm_unmatched = crm_silver.join(crm_email_matched, on="crm_customer_id", how="left_anti")
txn_unmatched = txn_silver.join(txn_email_matched, on="transaction_id", how="left_anti")

crm_fb_unique = (
    crm_unmatched
    .filter(F.col("email_normalized").isNull())
    .groupBy("fallback_key").count()
    .filter("count = 1")
    .drop("count")
)

txn_fb_unique = (
    txn_unmatched
    .filter(F.col("email_normalized").isNull())
    .groupBy("fallback_key").count()
    .filter("count = 1")
    .drop("count")
)

fallback_matches = (
    crm_unmatched.alias("crm")
    .join(
        crm_fb_unique.alias("cfb"),
        F.col("crm.fallback_key") == F.col("cfb.fallback_key"),
        "inner"
    )
    .join(
        txn_unmatched.alias("txn"),
        F.col("crm.fallback_key") == F.col("txn.fallback_key"),
        "inner"
    )
    .join(
        txn_fb_unique.alias("tfb"),
        F.col("txn.fallback_key") == F.col("tfb.fallback_key"),
        "inner"
    )
    .select(
        F.col("crm.crm_customer_id").alias("crm_customer_id"),
        F.col("crm.email_normalized").alias("crm_email_normalized"),
        F.col("crm.phone_normalized").alias("crm_phone_normalized"),
        F.col("crm.first_name_clean").alias("crm_first_name_clean"),
        F.col("crm.last_name_clean").alias("crm_last_name_clean"),
        F.col("crm.address_clean").alias("crm_address_clean"),
        F.col("crm.city_clean").alias("crm_city_clean"),
        F.col("crm.country_clean").alias("crm_country_clean"),
        F.col("crm.registration_date_clean").alias("crm_registration_date_clean"),
        F.col("crm.fallback_key").alias("fallback_key"),

        F.col("txn.transaction_id").alias("txn_transaction_id"),
        F.col("txn.transaction_email").alias("txn_transaction_email"),
        F.col("txn.email_normalized").alias("txn_email_normalized"),
        F.col("txn.phone_normalized").alias("txn_phone_normalized"),
        F.col("txn.first_name_clean").alias("txn_first_name_clean"),
        F.col("txn.last_name_clean").alias("txn_last_name_clean"),
        F.col("txn.address_clean").alias("txn_address_clean"),
        F.col("txn.city_clean").alias("txn_city_clean"),
        F.col("txn.country_clean").alias("txn_country_clean"),
        F.col("txn.purchase_date_clean").alias("txn_purchase_date_clean")
    )
)

print("CRM silver:", crm_silver.count())
print("TXN silver:", txn_silver.count())
print("Matched by email:", email_matches.count())
print("Matched by fallback:", fallback_matches.count())

fallback_matches.show(5, truncate=False)

CRM silver: 99
TXN silver: 79
Matched by email: 45
Matched by fallback: 8
+---------------+--------------------+--------------------+--------------------+-------------------+-----------------+--------------+-----------------+---------------------------+---------------------------------+------------------+---------------------+--------------------+--------------------+--------------------+-------------------+-----------------+--------------+-----------------+-----------------------+
|crm_customer_id|crm_email_normalized|crm_phone_normalized|crm_first_name_clean|crm_last_name_clean|crm_address_clean|crm_city_clean|crm_country_clean|crm_registration_date_clean|fallback_key                     |txn_transaction_id|txn_transaction_email|txn_email_normalized|txn_phone_normalized|txn_first_name_clean|txn_last_name_clean|txn_address_clean|txn_city_clean|txn_country_clean|txn_purchase_date_clean|
+---------------+--------------------+--------------------+--------------------+-------------------+

## Gold Layer — Final Golden Customer Record

In this layer we:
- merge matched CRM and transaction records
- retain CRM-only and transaction-only customers
- create a curated customer record with a consistent business schema
- add lineage columns such as match rule

In [22]:
golden_email = (
    email_matches
    .join(
        txn_metrics.alias("m"),
        (F.col("txn.email_normalized") == F.col("m.email_normalized")) &
        (F.col("txn.fallback_key") == F.col("m.fallback_key")),
        "left"
    )
    .select(
        F.sha2(
            F.concat_ws("||", F.col("crm.crm_customer_id"), F.col("crm.email_normalized")),
            256
        ).alias("golden_customer_id"),
        F.col("crm.crm_customer_id").alias("crm_customer_id"),
        F.coalesce(F.col("crm.first_name_clean"), F.col("txn.first_name_clean")).alias("first_name"),
        F.coalesce(F.col("crm.last_name_clean"), F.col("txn.last_name_clean")).alias("last_name"),
        F.coalesce(F.col("crm.email_normalized"), F.col("txn.email_normalized")).alias("email"),
        F.coalesce(F.col("crm.phone_normalized"), F.col("txn.phone_normalized")).alias("phone"),
        F.coalesce(F.col("crm.address_clean"), F.col("txn.address_clean")).alias("address"),
        F.coalesce(F.col("crm.city_clean"), F.col("txn.city_clean")).alias("city"),
        F.coalesce(F.col("crm.country_clean"), F.col("txn.country_clean")).alias("country"),
        F.col("crm.registration_date_clean").alias("registration_date"),
        F.col("m.first_purchase_date").alias("first_purchase_date"),
        F.col("m.last_purchase_date").alias("last_purchase_date"),
        F.col("m.transaction_count").alias("transaction_count"),
        F.lit("email_exact").alias("match_rule")
    )
)

golden_fallback = (
    fallback_matches.alias("x")
    .join(
        txn_metrics.alias("m"),
        F.col("x.fallback_key") == F.col("m.fallback_key"),
        "left"
    )
    .select(
        F.sha2(
            F.concat_ws("||", F.col("x.crm_customer_id"), F.col("x.fallback_key")),
            256
        ).alias("golden_customer_id"),
        F.col("x.crm_customer_id").alias("crm_customer_id"),
        F.coalesce(F.col("x.crm_first_name_clean"), F.col("x.txn_first_name_clean")).alias("first_name"),
        F.coalesce(F.col("x.crm_last_name_clean"), F.col("x.txn_last_name_clean")).alias("last_name"),
        F.coalesce(F.col("x.crm_email_normalized"), F.col("x.txn_email_normalized")).alias("email"),
        F.coalesce(F.col("x.crm_phone_normalized"), F.col("x.txn_phone_normalized")).alias("phone"),
        F.coalesce(F.col("x.crm_address_clean"), F.col("x.txn_address_clean")).alias("address"),
        F.coalesce(F.col("x.crm_city_clean"), F.col("x.txn_city_clean")).alias("city"),
        F.coalesce(F.col("x.crm_country_clean"), F.col("x.txn_country_clean")).alias("country"),
        F.col("x.crm_registration_date_clean").alias("registration_date"),
        F.col("m.first_purchase_date").alias("first_purchase_date"),
        F.col("m.last_purchase_date").alias("last_purchase_date"),
        F.col("m.transaction_count").alias("transaction_count"),
        F.lit("fallback_name_city_country").alias("match_rule")
    )
)

crm_matched_ids = (
    crm_email_matched
    .union(golden_fallback.select("crm_customer_id").distinct())
    .distinct()
)

txn_fallback_matched = (
    fallback_matches
    .select(F.col("txn_transaction_id").alias("transaction_id"))
    .distinct()
)

txn_matched_ids = (
    txn_email_matched
    .union(txn_fallback_matched)
    .distinct()
)

golden_crm_only = (
    crm_silver
    .join(crm_matched_ids, on="crm_customer_id", how="left_anti")
    .select(
        F.sha2(
            F.concat_ws("||", F.col("crm_customer_id"), F.col("email_normalized"), F.col("fallback_key")),
            256
        ).alias("golden_customer_id"),
        "crm_customer_id",
        F.col("first_name_clean").alias("first_name"),
        F.col("last_name_clean").alias("last_name"),
        F.col("email_normalized").alias("email"),
        F.col("phone_normalized").alias("phone"),
        F.col("address_clean").alias("address"),
        F.col("city_clean").alias("city"),
        F.col("country_clean").alias("country"),
        F.col("registration_date_clean").alias("registration_date"),
        F.lit(None).cast("date").alias("first_purchase_date"),
        F.lit(None).cast("date").alias("last_purchase_date"),
        F.lit(0).alias("transaction_count"),
        F.lit("crm_only").alias("match_rule")
    )
)

golden_txn_only = (
    txn_silver
    .join(txn_matched_ids, on="transaction_id", how="left_anti")
    .join(txn_metrics, on=["email_normalized", "fallback_key"], how="left")
    .select(
        F.sha2(
            F.concat_ws("||", F.col("email_normalized"), F.col("fallback_key"), F.col("transaction_id")),
            256
        ).alias("golden_customer_id"),
        F.lit(None).cast("string").alias("crm_customer_id"),
        F.col("first_name_clean").alias("first_name"),
        F.col("last_name_clean").alias("last_name"),
        F.col("email_normalized").alias("email"),
        F.col("phone_normalized").alias("phone"),
        F.col("address_clean").alias("address"),
        F.col("city_clean").alias("city"),
        F.col("country_clean").alias("country"),
        F.lit(None).cast("date").alias("registration_date"),
        F.col("first_purchase_date"),
        F.col("last_purchase_date"),
        F.col("transaction_count"),
        F.lit("txn_only").alias("match_rule")
    )
)

golden_customers = (
    golden_email
    .unionByName(golden_fallback)
    .unionByName(golden_crm_only)
    .unionByName(golden_txn_only)
    .dropDuplicates(["golden_customer_id"])
)

print("Gold records:", golden_customers.count())
golden_customers.show(20, truncate=False)

Gold records: 125
+----------------------------------------------------------------+---------------+----------+---------+--------------------------+------------+---------------+----------+-------+-----------------+-------------------+------------------+-----------------+--------------------------+
|golden_customer_id                                              |crm_customer_id|first_name|last_name|email                     |phone       |address        |city      |country|registration_date|first_purchase_date|last_purchase_date|transaction_count|match_rule                |
+----------------------------------------------------------------+---------------+----------+---------+--------------------------+------------+---------------+----------+-------+-----------------+-------------------+------------------+-----------------+--------------------------+
|02c3a5cde4bb87040c3c8bc50d4511cc104932feb14d2dab4116d4120f24401b|CRM00056       |Matthew   |Martin   |matthew.martin@email.com  |443524723

## Quality checks and export

Final step:
- generate pipeline metrics
- run basic validation
- export the Gold dataset

In [23]:
metrics = spark.createDataFrame(
    [
        ("bronze_crm_rows", crm_bronze.count()),
        ("bronze_txn_rows", txn_bronze.count()),
        ("silver_crm_rows", crm_silver.count()),
        ("silver_txn_rows", txn_silver.count()),
        ("gold_email_matches", golden_email.count()),
        ("gold_fallback_matches", golden_fallback.count()),
        ("gold_crm_only", golden_crm_only.count()),
        ("gold_txn_only", golden_txn_only.count()),
        ("gold_total_records", golden_customers.count()),
    ],
    ["metric_name", "metric_value"]
)

metrics.show(truncate=False)

duplicate_gold_ids = (
    golden_customers
    .groupBy("golden_customer_id")
    .count()
    .filter("count > 1")
    .count()
)

assert golden_customers.count() > 0, "Gold dataset is empty"
assert duplicate_gold_ids == 0, "golden_customer_id is not unique"

golden_customers.coalesce(1).write.mode("overwrite").option("header", True).csv("/content/gold_golden_customers")
metrics.coalesce(1).write.mode("overwrite").option("header", True).csv("/content/gold_metrics")

print("Validation passed")
print("Exported:")
print("/content/gold_golden_customers")
print("/content/gold_metrics")

+---------------------+------------+
|metric_name          |metric_value|
+---------------------+------------+
|bronze_crm_rows      |108         |
|bronze_txn_rows      |80          |
|silver_crm_rows      |99          |
|silver_txn_rows      |79          |
|gold_email_matches   |45          |
|gold_fallback_matches|8           |
|gold_crm_only        |46          |
|gold_txn_only        |26          |
|gold_total_records   |125         |
+---------------------+------------+

Validation passed
Exported:
/content/gold_golden_customers
/content/gold_metrics
